# Virginia Regional Economic Intelligence

## 02 — Data Quality and Analytical Readiness

This notebook evaluates the quality and usability of the longitudinal Virginia QCEW industry panel before regional economic indicators are calculated.

### Objectives

- quantify missing and disclosure-suppressed observations,
- identify which industries and localities are most affected by suppression,
- evaluate temporal continuity within locality-industry series,
- test internal consistency among employment and wage measures,
- identify potentially anomalous year-to-year changes,
- and define which observations are suitable for longitudinal economic analysis.

The purpose of these checks is not to remove unusual economic observations automatically. Instead, the analysis distinguishes between genuine economic variation, confidentiality-related missingness, and potential data-quality concerns.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
# Project paths

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES = PROJECT_ROOT / "outputs" / "figures"

PANEL_PATH = (
    PROCESSED_DIR
    / "va_qcew_private_sector_panel_2019_2025.csv"
)

print(f"Panel source: {PANEL_PATH}")

Panel source: C:\Users\Chris\Documents\virginia-regional-economic-intelligence\data\processed\va_qcew_private_sector_panel_2019_2025.csv


In [3]:
# Load the validated output produced in Notebook 01.
# --------------------------------------------------
# Identifier fields are explicitly retained as strings so that FIPS and
# industry codes remain categorical identifiers rather than quantities.

panel = pd.read_csv(
    PANEL_PATH,
    dtype={
        "area_fips": str,
        "industry_code": str,
        "own_code": str,
        "agglvl_code": str,
        "disclosure_code": str
    }
)

print(f"Rows: {len(panel):,}")
print(f"Columns: {panel.shape[1]}")
print(
    f"Study period: "
    f"{panel['year'].min()}–{panel['year'].max()}"
)

panel.head()

Rows: 17,804
Columns: 13
Study period: 2019–2025


,year,area_fips,locality_name,industry_code,industry_title,own_code,agglvl_code,disclosure_code,annual_avg_estabs,annual_avg_emplvl,total_annual_wages,annual_avg_wkly_wage,avg_annual_pay
0,2019,51001,Accomack County,11,"Agriculture, Forestry, Fishing and Hunting",5,74,NaN,21,150.0,7084971.0,910.0,47338.0
1,2020,51001,Accomack County,11,"Agriculture, Forestry, Fishing and Hunting",5,74,NaN,22,150.0,8339362.0,1073.0,55782.0
2,2021,51001,Accomack County,11,"Agriculture, Forestry, Fishing and Hunting",5,74,NaN,23,149.0,8779149.0,1136.0,59053.0
3,2022,51001,Accomack County,11,"Agriculture, Forestry, Fishing and Hunting",5,74,N,30,NaN,NaN,NaN,NaN
4,2023,51001,Accomack County,11,"Agriculture, Forestry, Fishing and Hunting",5,74,N,33,NaN,NaN,NaN,NaN


## 1. Revalidate the Processed Input

Although the source dataset was validated during construction, key structural checks are repeated after loading the saved file.

This confirms that serialization to CSV did not alter identifiers, introduce duplicate observations, or change the expected analytical population.

In [4]:
# Revalidate the expected analytical structure.

input_validation = {
    "records": len(panel),
    "years": panel["year"].nunique(),
    "localities": panel["area_fips"].nunique(),
    "sectors": panel["industry_code"].nunique(),
    "duplicate_keys": panel.duplicated(
        subset=["area_fips", "industry_code", "year"]
    ).sum(),
    "missing_locality_names": panel["locality_name"].isna().sum(),
    "missing_industry_titles": panel["industry_title"].isna().sum()
}

pd.Series(input_validation, name="value")

records                    17804
years                          7
localities                   133
sectors                       20
duplicate_keys                 0
missing_locality_names         0
missing_industry_titles        0
Name: value, dtype: int64

In [5]:
# Assert the structural characteristics established in Notebook 01.

assert input_validation["records"] == 17_804
assert input_validation["years"] == 7
assert input_validation["localities"] == 133
assert input_validation["sectors"] == 20
assert input_validation["duplicate_keys"] == 0
assert input_validation["missing_locality_names"] == 0
assert input_validation["missing_industry_titles"] == 0

print("Processed input validation passed.")

Processed input validation passed.


## 2. Disclosure Suppression and Missingness

QCEW confidentiality rules can prevent publication of employment and wage information for individual locality-industry observations.

These suppressed records remain useful for identifying the existence of an establishment-industry combination, but unavailable employment and wage values cannot be used directly in calculations such as growth rates, industry shares, or location quotients.

The first data-quality assessment therefore measures the distribution of suppression across time, industries, and geographic areas.

In [6]:
# Create an explicit suppression indicator.

panel["is_suppressed"] = (
    panel["disclosure_code"] == "N"
)

print(
    f"Suppressed records: "
    f"{panel['is_suppressed'].sum():,}"
)

print(
    f"Overall suppression rate: "
    f"{panel['is_suppressed'].mean():.1%}"
)

Suppressed records: 5,289
Overall suppression rate: 29.7%


In [7]:
# Summarize suppression by year.

suppression_by_year = (
    panel
    .groupby("year")
    .agg(
        total_records=("is_suppressed", "size"),
        suppressed_records=("is_suppressed", "sum"),
        suppression_rate=("is_suppressed", "mean")
    )
    .reset_index()
)

suppression_by_year

,year,total_records,suppressed_records,suppression_rate
0,2019,2522,677,0.268438
1,2020,2521,690,0.273701
2,2021,2524,695,0.275357
3,2022,2559,702,0.274326
4,2023,2572,738,0.286936
5,2024,2556,806,0.315336
6,2025,2550,981,0.384706


### Initial Suppression Pattern

Disclosure suppression is not constant across the study period.

The suppression rate increases materially toward the end of the panel, meaning that apparent changes in sector coverage may partly reflect confidentiality constraints rather than economic disappearance.

For that reason, later trend calculations will require published values at the relevant comparison dates rather than assuming missing or suppressed observations represent zero employment.

In [8]:
# Measure suppression by industry across the full study period.

suppression_by_sector = (
    panel
    .groupby(
        ["industry_code", "industry_title"]
    )
    .agg(
        records=("is_suppressed", "size"),
        suppressed_records=("is_suppressed", "sum"),
        suppression_rate=("is_suppressed", "mean")
    )
    .reset_index()
    .sort_values(
        "suppression_rate",
        ascending=False
    )
)

suppression_by_sector

,industry_code,industry_title,records,suppressed_records,suppression_rate
1,21,"Mining, Quarrying, and Oil and Gas Extraction",593,470,0.792580
2,22,Utilities,727,549,0.755158
0,11,"Agriculture, Forestry, Fishing and Hunting",854,570,0.667447
12,55,Management of Companies and Enterprises,850,404,0.475294
14,61,Educational Services,885,371,0.419209
15,62,Health Care and Social Assistance,931,372,0.399570
5,42,Wholesale Trade,930,344,0.369892
7,48-49,Transportation and Warehousing,923,335,0.362947
8,51,Information,910,275,0.302198
19,99,Unclassified,931,271,0.291085


### Industry-Level Suppression Pattern

Disclosure suppression varies substantially across industries.

Mining, utilities, agriculture, and management of companies have the highest suppression rates, while broadly distributed sectors such as retail trade and other services have comparatively low suppression.

This pattern is consistent with the confidentiality challenges associated with small or highly concentrated local industries. A locality with only one or a few employers in a sector is more likely to have employment and wage values withheld.

The uneven distribution of suppression has an important analytical implication: industries with high confidentiality rates will have smaller usable samples for longitudinal comparisons. Subsequent regional rankings and growth calculations must therefore report the number of valid observations and avoid treating suppressed records as zeros.

## 3. Geographic Distribution of Disclosure Suppression

Disclosure suppression may also vary systematically across Virginia localities.

Smaller or less economically diversified counties may contain fewer employers within individual industries, increasing the likelihood that publication would reveal confidential information.

Suppression rates are therefore calculated for each county and independent city across the full study period.

In [9]:
# Measure suppression across Virginia localities.

suppression_by_locality = (
    panel
    .groupby(
        ["area_fips", "locality_name"]
    )
    .agg(
        records=("is_suppressed", "size"),
        suppressed_records=("is_suppressed", "sum"),
        suppression_rate=("is_suppressed", "mean")
    )
    .reset_index()
    .sort_values(
        "suppression_rate",
        ascending=False
    )
)

suppression_by_locality.head(20)

,area_fips,locality_name,records,suppressed_records,suppression_rate
22,51045,Craig County,127,104,0.818898
10,51021,Bland County,125,94,0.752000
86,51181,Surry County,127,84,0.661417
39,51081,Greensville County,134,87,0.649254
18,51036,Charles City County,140,87,0.621429
37,51077,Grayson County,140,86,0.614286
112,51678,Lexington city,111,68,0.612613
8,51017,Bath County,131,79,0.603053
83,51175,Southampton County,138,80,0.579710
97,51530,Buena Vista city,118,68,0.576271


In [10]:
# Identify localities with the lowest suppression rates.

suppression_by_locality.tail(20)

,area_fips,locality_name,records,suppressed_records,suppression_rate
36,51075,Goochland County,133,18,0.135338
7,51015,Augusta County,140,18,0.128571
23,51047,Culpeper County,140,18,0.128571
68,51143,Pittsylvania County,140,17,0.121429
15,51031,Campbell County,140,16,0.114286
52,51107,Loudoun County,140,16,0.114286
129,51810,Virginia Beach city,136,15,0.110294
9,51019,Bedford County,137,14,0.102190
41,51085,Hanover County,140,11,0.078571
42,51087,Henrico County,140,11,0.078571


In [11]:
# Summarize geographic variation in suppression.

suppression_by_locality["suppression_rate"].describe()

count    133.000000
mean       0.300266
std        0.172725
min        0.000000
25%        0.171642
50%        0.285714
75%        0.416058
max        0.818898
Name: suppression_rate, dtype: float64

In [12]:
# Estimate establishment presence by locality.
# Establishment counts provide a rough measure of local economic scale
# and diversification, even when employment values are suppressed.

locality_establishment_summary = (
    panel
    .groupby(
        ["area_fips", "locality_name"]
    )
    .agg(
        total_sector_establishments=(
            "annual_avg_estabs",
            "sum"
        )
    )
    .reset_index()
)

locality_quality = (
    suppression_by_locality
    .merge(
        locality_establishment_summary,
        on=["area_fips", "locality_name"],
        how="left",
        validate="one_to_one"
    )
)

suppression_establishment_corr = (
    locality_quality[
        [
            "suppression_rate",
            "total_sector_establishments"
        ]
    ]
    .corr()
    .iloc[0, 1]
)

print(
    "Correlation between suppression rate and "
    f"establishment count: {suppression_establishment_corr:.3f}"
)

Correlation between suppression rate and establishment count: -0.447


### Geographic Suppression and Economic Scale

The correlation between locality suppression rates and total establishment counts is **-0.447**.

This indicates a moderate negative relationship: localities with larger establishment bases tend to have lower disclosure-suppression rates. This pattern is consistent with confidentiality constraints being more binding in smaller or less economically diversified local economies.

The relationship is descriptive and should not be interpreted as causal.

## 4. Temporal Completeness of Locality-Industry Series

Longitudinal economic analysis requires repeated published observations across time.

For each locality-industry combination, the number of years with available employment data is calculated. This helps identify which series can reliably support growth and trend analysis and which are too heavily affected by confidentiality suppression.

In [13]:
# Measure the number of usable employment observations
# for each locality × industry time series.

series_quality = (
    panel
    .groupby(
        [
            "area_fips",
            "locality_name",
            "industry_code",
            "industry_title"
        ]
    )
    .agg(
        years_present=("year", "nunique"),
        published_employment_years=(
            "annual_avg_emplvl",
            "count"
        ),
        suppressed_years=(
            "is_suppressed",
            "sum"
        )
    )
    .reset_index()
)

series_quality.head()

,area_fips,locality_name,industry_code,industry_title,years_present,published_employment_years,suppressed_years
0,51001,Accomack County,11,"Agriculture, Forestry, Fishing and Hunting",7,5,2
1,51001,Accomack County,21,"Mining, Quarrying, and Oil and Gas Extraction",2,0,2
2,51001,Accomack County,22,Utilities,7,0,7
3,51001,Accomack County,23,Construction,7,7,0
4,51001,Accomack County,31-33,Manufacturing,7,7,0


In [14]:
# Summarize the number of published employment years
# available for each locality × industry series.

published_year_distribution = (
    series_quality["published_employment_years"]
    .value_counts()
    .sort_index()
    .rename_axis("published_employment_years")
    .reset_index(name="series_count")
)

published_year_distribution

,published_employment_years,series_count
0,0,526
1,1,83
2,2,72
3,3,89
4,4,92
5,5,114
6,6,243
7,7,1375


### Temporal Completeness Pattern

Time-series completeness varies substantially across locality-industry combinations.

A large share of series are highly usable for longitudinal analysis, including 1,375 series with published employment in all seven years and 243 with six published years.

However, 526 locality-industry series have no published employment observations during the study period. These series cannot support employment-growth analysis and should be excluded from longitudinal calculations while remaining documented as part of the underlying QCEW coverage.

In [15]:
# Classify each locality × industry series by temporal completeness.

def classify_series_quality(published_years):
    """
    Assign an analytical-readiness category based on the number
    of years with published employment data.
    """

    if published_years == 7:
        return "Complete"
    elif published_years >= 5:
        return "High coverage"
    elif published_years >= 3:
        return "Moderate coverage"
    else:
        return "Limited coverage"


series_quality["quality_category"] = (
    series_quality["published_employment_years"]
    .apply(classify_series_quality)
)

quality_category_counts = (
    series_quality["quality_category"]
    .value_counts()
    .reindex(
        [
            "Complete",
            "High coverage",
            "Moderate coverage",
            "Limited coverage"
        ]
    )
    .rename_axis("quality_category")
    .reset_index(name="series_count")
)

quality_category_counts

,quality_category,series_count
0,Complete,1375
1,High coverage,357
2,Moderate coverage,181
3,Limited coverage,681


### Analytical Readiness Categories

Locality-industry series are grouped by the number of years with published employment data.

- **Complete:** all 7 years published
- **High coverage:** 5–6 years published
- **Moderate coverage:** 3–4 years published
- **Limited coverage:** fewer than 3 years published

These categories will be used later to determine which series are appropriate for trend analysis, growth calculations, and regional comparisons.

In [16]:
# Identify series with published employment at both study endpoints.
# ---------------------------------------------------------------
# Full-period growth from 2019 to 2025 can only be calculated when
# employment is available in both years.

endpoint_availability = (
    panel[
        panel["year"].isin([2019, 2025])
    ]
    .pivot_table(
        index=[
            "area_fips",
            "locality_name",
            "industry_code",
            "industry_title"
        ],
        columns="year",
        values="annual_avg_emplvl",
        aggfunc="first"
    )
    .reset_index()
)

endpoint_availability["has_2019_and_2025"] = (
    endpoint_availability[2019].notna()
    & endpoint_availability[2025].notna()
)

endpoint_summary = (
    endpoint_availability["has_2019_and_2025"]
    .value_counts()
    .rename_axis("has_both_endpoints")
    .reset_index(name="series_count")
)

endpoint_summary

,has_both_endpoints,series_count
0,True,1459
1,False,496


In [17]:
# Rebuild endpoint availability using the full locality × industry universe.
# This ensures that series missing both endpoint years are still represented.

endpoint_availability = (
    series_quality[
        [
            "area_fips",
            "locality_name",
            "industry_code",
            "industry_title"
        ]
    ]
    .merge(
        endpoint_availability,
        on=[
            "area_fips",
            "locality_name",
            "industry_code",
            "industry_title"
        ],
        how="left",
        validate="one_to_one"
    )
)

endpoint_availability["has_2019_and_2025"] = (
    endpoint_availability[2019].notna()
    & endpoint_availability[2025].notna()
)

endpoint_summary = (
    endpoint_availability["has_2019_and_2025"]
    .value_counts()
    .rename_axis("has_both_endpoints")
    .reset_index(name="series_count")
)

endpoint_summary

,has_both_endpoints,series_count
0,True,1459
1,False,1135


### Endpoint Availability for Long-Term Growth

Of the 2,594 locality-industry series in the panel, 1,459 have published employment values in both 2019 and 2025.

These series can support direct full-period employment growth calculations. Series missing either endpoint are excluded from 2019–2025 growth rankings because suppressed or unavailable values should not be treated as zero employment.

In [18]:
# Calculate the share of locality-industry series that can support
# a direct 2019-to-2025 employment growth comparison.

endpoint_coverage_rate = (
    endpoint_availability["has_2019_and_2025"].mean()
)

print(
    f"Series with both 2019 and 2025 employment available: "
    f"{endpoint_coverage_rate:.1%}"
)

Series with both 2019 and 2025 employment available: 56.2%


### Full-Period Growth Coverage

Approximately **56.2%** of locality-industry series have published employment values in both 2019 and 2025.

These series form the valid sample for direct full-period employment growth analysis. The remaining series are excluded from 2019–2025 growth calculations because one or both endpoint values are unavailable.

In [19]:
# Calculate full-period employment growth for series with both endpoints.
# ----------------------------------------------------------------------
# Only locality × industry series with published employment in both
# 2019 and 2025 are included in the calculation.

employment_growth_2019_2025 = (
    endpoint_availability[
        endpoint_availability["has_2019_and_2025"]
    ]
    .copy()
)

employment_growth_2019_2025["employment_change"] = (
    employment_growth_2019_2025[2025]
    - employment_growth_2019_2025[2019]
)

employment_growth_2019_2025["employment_growth_pct"] = (
    employment_growth_2019_2025["employment_change"]
    / employment_growth_2019_2025[2019]
)

employment_growth_2019_2025[
    [
        "area_fips",
        "locality_name",
        "industry_code",
        "industry_title",
        2019,
        2025,
        "employment_change",
        "employment_growth_pct"
    ]
].head()

,area_fips,locality_name,industry_code,industry_title,2019,2025,employment_change,employment_growth_pct
0,51001,Accomack County,11,"Agriculture, Forestry, Fishing and Hunting",150.0,199.0,49.0,0.326667
3,51001,Accomack County,23,Construction,391.0,349.0,-42.0,-0.107417
4,51001,Accomack County,31-33,Manufacturing,3285.0,3229.0,-56.0,-0.017047
5,51001,Accomack County,42,Wholesale Trade,240.0,167.0,-73.0,-0.304167
6,51001,Accomack County,44-45,Retail Trade,1300.0,1176.0,-124.0,-0.095385


In [20]:
# Check whether any valid endpoint series begin with zero employment.
# Percentage growth is not meaningful when the baseline value is zero.

zero_baseline_count = (
    employment_growth_2019_2025[2019] == 0
).sum()

print(
    f"Series with zero employment in 2019: "
    f"{zero_baseline_count}"
)

Series with zero employment in 2019: 0


In [21]:
# Inspect the largest 2019–2025 employment growth rates.
# Extreme percentage changes may reflect very small baseline employment,
# so they should be reviewed before being used in rankings.

largest_growth_rates = (
    employment_growth_2019_2025
    .sort_values(
        "employment_growth_pct",
        ascending=False
    )
    [
        [
            "locality_name",
            "industry_title",
            2019,
            2025,
            "employment_change",
            "employment_growth_pct"
        ]
    ]
    .head(20)
)

largest_growth_rates

,locality_name,industry_title,2019,2025,employment_change,employment_growth_pct
2396,Portsmouth city,"Arts, Entertainment, and Recreation",140.0,1365.0,1225.0,8.750000
1088,Madison County,Transportation and Warehousing,11.0,71.0,60.0,5.454545
2544,Waynesboro city,Transportation and Warehousing,185.0,980.0,795.0,4.297297
1524,Rockbridge County,Real Estate and Rental and Leasing,21.0,110.0,89.0,4.238095
192,Bedford County,Educational Services,47.0,211.0,164.0,3.489362
1001,Lancaster County,"Arts, Entertainment, and Recreation",28.0,123.0,95.0,3.392857
2504,Suffolk city,Transportation and Warehousing,2323.0,9944.0,7621.0,3.280672
768,Greene County,Unclassified,4.0,17.0,13.0,3.250000
1681,Stafford County,Transportation and Warehousing,1581.0,6230.0,4649.0,2.940544
1831,Wise County,Unclassified,9.0,34.0,25.0,2.777778


### Small-Base Effects in Percentage Growth

Percentage growth can become misleading when the starting employment base is very small.

Several of the largest observed growth rates originate from industries with only a few dozen employees in 2019. While these changes are mathematically valid, they can disproportionately dominate rankings without representing the largest economically significant changes.

For comparative growth rankings, a minimum baseline employment threshold of **100 employees in 2019** is therefore applied. Absolute employment change remains available separately so that smaller industries are not removed from the dataset entirely.

In [22]:
# Apply a minimum baseline-employment threshold to growth rankings.

MIN_BASE_EMPLOYMENT = 100

growth_rank_eligible = (
    employment_growth_2019_2025[
        employment_growth_2019_2025[2019] >= MIN_BASE_EMPLOYMENT
    ]
    .copy()
)

print(
    f"Series eligible for percentage-growth rankings: "
    f"{len(growth_rank_eligible):,}"
)

print(
    f"Share of endpoint-complete series retained: "
    f"{len(growth_rank_eligible) / len(employment_growth_2019_2025):.1%}"
)

Series eligible for percentage-growth rankings: 1,145
Share of endpoint-complete series retained: 78.5%


In [23]:
# Review the largest employment growth rates after applying
# the minimum baseline-employment threshold.

top_growth_rankings = (
    growth_rank_eligible
    .sort_values(
        "employment_growth_pct",
        ascending=False
    )
    [
        [
            "locality_name",
            "industry_title",
            2019,
            2025,
            "employment_change",
            "employment_growth_pct"
        ]
    ]
    .head(20)
)

top_growth_rankings

,locality_name,industry_title,2019,2025,employment_change,employment_growth_pct
2396,Portsmouth city,"Arts, Entertainment, and Recreation",140.0,1365.0,1225.0,8.750000
2544,Waynesboro city,Transportation and Warehousing,185.0,980.0,795.0,4.297297
2504,Suffolk city,Transportation and Warehousing,2323.0,9944.0,7621.0,3.280672
1681,Stafford County,Transportation and Warehousing,1581.0,6230.0,4649.0,2.940544
364,Charles City County,Transportation and Warehousing,149.0,496.0,347.0,2.328859
662,Frederick County,Management of Companies and Enterprises,165.0,444.0,279.0,1.690909
1269,Nottoway County,"Professional, Scientific, and Technical Services",105.0,277.0,172.0,1.638095
130,Arlington County,Management of Companies and Enterprises,3159.0,7881.0,4722.0,1.494777
2455,Roanoke city,"Arts, Entertainment, and Recreation",653.0,1624.0,971.0,1.486983
2429,Richmond city,Real Estate and Rental and Leasing,2094.0,5147.0,3053.0,1.457975


In [24]:
# Review the largest employment declines among series that meet
# the minimum baseline-employment threshold.

largest_declines = (
    growth_rank_eligible
    .sort_values(
        "employment_growth_pct",
        ascending=True
    )
    [
        [
            "locality_name",
            "industry_title",
            2019,
            2025,
            "employment_change",
            "employment_growth_pct"
        ]
    ]
    .head(20)
)

largest_declines

,locality_name,industry_title,2019,2025,employment_change,employment_growth_pct
2071,Falls Church city,Administrative and Support and Waste Management,1127.0,192.0,-935.0,-0.829636
919,James City County,Management of Companies and Enterprises,1013.0,224.0,-789.0,-0.778875
552,Fairfax County,"Mining, Quarrying, and Oil and Gas Extraction",269.0,69.0,-200.0,-0.743494
1327,Patrick County,"Professional, Scientific, and Technical Services",285.0,77.0,-208.0,-0.729825
261,Buchanan County,Construction,347.0,106.0,-241.0,-0.694524
1485,Richmond County,"Professional, Scientific, and Technical Services",126.0,40.0,-86.0,-0.682540
1410,Prince George County,Educational Services,170.0,54.0,-116.0,-0.682353
821,Hanover County,Management of Companies and Enterprises,1390.0,444.0,-946.0,-0.680576
1820,Wise County,Information,143.0,46.0,-97.0,-0.678322
931,King and Queen County,Manufacturing,141.0,47.0,-94.0,-0.666667


In [25]:
# Calculate year-over-year employment changes within each
# locality × industry time series.

panel = panel.sort_values(
    ["area_fips", "industry_code", "year"]
).copy()

panel["employment_change_yoy"] = (
    panel
    .groupby(["area_fips", "industry_code"])["annual_avg_emplvl"]
    .diff()
)

panel["employment_growth_yoy"] = (
    panel
    .groupby(["area_fips", "industry_code"])["annual_avg_emplvl"]
    .pct_change(fill_method=None)
)

panel[
    [
        "year",
        "locality_name",
        "industry_title",
        "annual_avg_emplvl",
        "employment_change_yoy",
        "employment_growth_yoy"
    ]
].head(15)

,year,locality_name,industry_title,annual_avg_emplvl,employment_change_yoy,employment_growth_yoy
0,2019,Accomack County,"Agriculture, Forestry, Fishing and Hunting",150.0,NaN,NaN
1,2020,Accomack County,"Agriculture, Forestry, Fishing and Hunting",150.0,0.0,0.000000
2,2021,Accomack County,"Agriculture, Forestry, Fishing and Hunting",149.0,-1.0,-0.006667
3,2022,Accomack County,"Agriculture, Forestry, Fishing and Hunting",NaN,NaN,NaN
4,2023,Accomack County,"Agriculture, Forestry, Fishing and Hunting",NaN,NaN,NaN
5,2024,Accomack County,"Agriculture, Forestry, Fishing and Hunting",193.0,NaN,NaN
6,2025,Accomack County,"Agriculture, Forestry, Fishing and Hunting",199.0,6.0,0.031088
7,2022,Accomack County,"Mining, Quarrying, and Oil and Gas Extraction",NaN,NaN,NaN
8,2023,Accomack County,"Mining, Quarrying, and Oil and Gas Extraction",NaN,NaN,NaN
9,2019,Accomack County,Utilities,NaN,NaN,NaN


In [26]:
# Inspect the most extreme year-over-year employment changes.
# Only observations with valid consecutive-year employment values
# contribute to the ranking.

largest_yoy_changes = (
    panel[
        panel["employment_growth_yoy"].notna()
    ]
    .assign(
        abs_employment_growth_yoy=lambda x:
            x["employment_growth_yoy"].abs()
    )
    .sort_values(
        "abs_employment_growth_yoy",
        ascending=False
    )
    [
        [
            "year",
            "locality_name",
            "industry_title",
            "annual_avg_emplvl",
            "employment_change_yoy",
            "employment_growth_yoy"
        ]
    ]
    .head(20)
)

largest_yoy_changes

,year,locality_name,industry_title,annual_avg_emplvl,employment_change_yoy,employment_growth_yoy
11771,2020,Surry County,Unclassified,1.0,1.0,inf
12589,2022,Wise County,Unclassified,32.0,29.0,9.666667
4051,2022,Fauquier County,Unclassified,61.0,53.0,6.625000
130,2021,Accomack County,Unclassified,7.0,6.0,6.000000
14640,2022,Galax city,Unclassified,43.0,35.0,4.375000
13253,2023,Buena Vista city,Unclassified,10.0,8.0,4.000000
13113,2022,Bristol city,"Arts, Entertainment, and Recreation",83.0,66.0,3.882353
10946,2020,Scott County,Unclassified,19.0,15.0,3.750000
10479,2022,Rockbridge County,Real Estate and Rental and Leasing,84.0,66.0,3.666667
17626,2023,Williamsburg city,Management of Companies and Enterprises,440.0,344.0,3.583333


In [27]:
# Create the prior-year employment value explicitly so that
# year-over-year changes can be screened for small-base effects.

panel["prior_year_employment"] = (
    panel
    .groupby(["area_fips", "industry_code"])["annual_avg_emplvl"]
    .shift(1)
)

# Retain year-over-year comparisons only when both consecutive years
# are published and the prior-year employment base is at least 100.

yoy_analysis_eligible = panel[
    panel["employment_growth_yoy"].notna()
    & np.isfinite(panel["employment_growth_yoy"])
    & (panel["prior_year_employment"] >= 100)
].copy()

print(
    f"Year-over-year comparisons eligible for anomaly review: "
    f"{len(yoy_analysis_eligible):,}"
)

Year-over-year comparisons eligible for anomaly review: 7,568


In [28]:
# Review the most extreme year-over-year changes after applying
# the minimum baseline-employment threshold.

largest_eligible_yoy_changes = (
    yoy_analysis_eligible
    .assign(
        abs_employment_growth_yoy=lambda x:
            x["employment_growth_yoy"].abs()
    )
    .sort_values(
        "abs_employment_growth_yoy",
        ascending=False
    )
    [
        [
            "year",
            "locality_name",
            "industry_title",
            "prior_year_employment",
            "annual_avg_emplvl",
            "employment_change_yoy",
            "employment_growth_yoy"
        ]
    ]
    .head(20)
)

largest_eligible_yoy_changes

,year,locality_name,industry_title,prior_year_employment,annual_avg_emplvl,employment_change_yoy,employment_growth_yoy
16444,2023,Portsmouth city,"Arts, Entertainment, and Recreation",275.0,1222.0,947.0,3.443636
2517,2025,Charles City County,Transportation and Warehousing,136.0,496.0,360.0,2.647059
8716,2021,Nottoway County,"Professional, Scientific, and Technical Services",112.0,395.0,283.0,2.526786
4543,2020,Frederick County,Management of Companies and Enterprises,165.0,574.0,409.0,2.478788
17198,2025,Suffolk city,Transportation and Warehousing,2868.0,9944.0,7076.0,2.467225
11569,2022,Stafford County,Information,179.0,594.0,415.0,2.318436
6791,2023,Lancaster County,Manufacturing,113.0,338.0,225.0,1.991150
11565,2025,Stafford County,Transportation and Warehousing,2136.0,6230.0,4094.0,1.916667
2277,2023,Caroline County,Administrative and Support and Waste Management,244.0,683.0,439.0,1.799180
17463,2025,Waynesboro city,Transportation and Warehousing,364.0,980.0,616.0,1.692308


In [29]:
# Flag unusually large year-over-year employment changes for review.
# These observations are not removed; they are simply marked as potential
# anomalies that may reflect genuine economic events, reclassification,
# establishment openings/closures, or data issues.

YOY_GROWTH_FLAG_THRESHOLD = 1.00  # ±100%

yoy_analysis_eligible["large_yoy_change_flag"] = (
    yoy_analysis_eligible["employment_growth_yoy"].abs()
    >= YOY_GROWTH_FLAG_THRESHOLD
)

flagged_yoy_changes = (
    yoy_analysis_eligible[
        yoy_analysis_eligible["large_yoy_change_flag"]
    ]
    .copy()
)

print(
    f"Year-over-year changes flagged for review: "
    f"{len(flagged_yoy_changes):,}"
)

print(
    f"Share of eligible comparisons flagged: "
    f"{len(flagged_yoy_changes) / len(yoy_analysis_eligible):.1%}"
)

Year-over-year changes flagged for review: 13
Share of eligible comparisons flagged: 0.2%


### Extreme Year-over-Year Changes

Only 13 of 7,568 eligible year-over-year employment comparisons exceed an absolute change of 100%, representing approximately **0.2%** of the qualified sample.

These observations are retained and flagged for review rather than removed automatically. Large changes may reflect genuine economic events, establishment openings or closures, industry reclassification, or other structural changes in local economies.

In [30]:
flagged_yoy_changes[
    [
        "year",
        "locality_name",
        "industry_title",
        "prior_year_employment",
        "annual_avg_emplvl",
        "employment_change_yoy",
        "employment_growth_yoy"
    ]
].sort_values(
    "employment_growth_yoy",
    ascending=False
)

,year,locality_name,industry_title,prior_year_employment,annual_avg_emplvl,employment_change_yoy,employment_growth_yoy
16444,2023,Portsmouth city,"Arts, Entertainment, and Recreation",275.0,1222.0,947.0,3.443636
2517,2025,Charles City County,Transportation and Warehousing,136.0,496.0,360.0,2.647059
8716,2021,Nottoway County,"Professional, Scientific, and Technical Services",112.0,395.0,283.0,2.526786
4543,2020,Frederick County,Management of Companies and Enterprises,165.0,574.0,409.0,2.478788
17198,2025,Suffolk city,Transportation and Warehousing,2868.0,9944.0,7076.0,2.467225
11569,2022,Stafford County,Information,179.0,594.0,415.0,2.318436
6791,2023,Lancaster County,Manufacturing,113.0,338.0,225.0,1.991150
11565,2025,Stafford County,Transportation and Warehousing,2136.0,6230.0,4094.0,1.916667
2277,2023,Caroline County,Administrative and Support and Waste Management,244.0,683.0,439.0,1.799180
17463,2025,Waynesboro city,Transportation and Warehousing,364.0,980.0,616.0,1.692308


## 5. Internal Consistency of Wage Measures

The QCEW provides both average weekly wages and average annual pay.

As a data-quality check, reported annual pay is compared with annualized weekly wages. The two measures should be closely aligned, although small differences may occur because of rounding or the specific averaging procedures used in the source data.

Large discrepancies may indicate records requiring additional review.

In [31]:
# Compare reported annual pay with annualized weekly wages.

wage_check = panel[
    panel["annual_avg_wkly_wage"].notna()
    & panel["avg_annual_pay"].notna()
].copy()

wage_check["annual_pay_from_weekly"] = (
    wage_check["annual_avg_wkly_wage"] * 52
)

wage_check["annual_pay_difference"] = (
    wage_check["avg_annual_pay"]
    - wage_check["annual_pay_from_weekly"]
)

wage_check["annual_pay_pct_difference"] = (
    wage_check["annual_pay_difference"]
    / wage_check["avg_annual_pay"]
)

wage_check[
    "annual_pay_pct_difference"
].describe()

count    1.251400e+04
mean    -4.721478e-07
std      4.123260e-04
min     -1.887386e-03
25%     -2.524671e-04
50%      0.000000e+00
75%      2.598077e-04
max      1.945869e-03
Name: annual_pay_pct_difference, dtype: float64

### Wage Consistency Check

Reported average annual pay is highly consistent with average weekly wages annualized over 52 weeks.

The median percentage difference is effectively zero, and even the largest discrepancies are below approximately 0.2%. These small differences are consistent with rounding and do not indicate a material data-quality problem.

In [32]:
# Compare reported average annual pay with the value implied by
# total annual wages divided by average annual employment.

pay_identity_check = panel[
    panel["total_annual_wages"].notna()
    & panel["annual_avg_emplvl"].notna()
    & (panel["annual_avg_emplvl"] > 0)
    & panel["avg_annual_pay"].notna()
].copy()

pay_identity_check["implied_avg_annual_pay"] = (
    pay_identity_check["total_annual_wages"]
    / pay_identity_check["annual_avg_emplvl"]
)

pay_identity_check["avg_pay_pct_difference"] = (
    pay_identity_check["avg_annual_pay"]
    - pay_identity_check["implied_avg_annual_pay"]
) / pay_identity_check["avg_annual_pay"]

pay_identity_check[
    "avg_pay_pct_difference"
].describe()

count    12512.000000
mean         0.000395
std          0.009071
min         -0.250024
25%         -0.000470
50%          0.000007
75%          0.000805
max          0.166665
Name: avg_pay_pct_difference, dtype: float64

### Annual Pay Identity Check

For most published observations, reported average annual pay closely matches the value implied by total annual wages divided by average annual employment.

The median difference is effectively zero, indicating strong overall internal consistency. A small number of observations show substantially larger discrepancies and are examined separately before determining whether they represent data-quality concerns or small-employment rounding effects.

In [33]:
# Inspect observations with the largest discrepancies between
# reported and implied average annual pay.

largest_pay_discrepancies = (
    pay_identity_check
    .assign(
        abs_avg_pay_pct_difference=lambda x:
            x["avg_pay_pct_difference"].abs()
    )
    .sort_values(
        "abs_avg_pay_pct_difference",
        ascending=False
    )
    [
        [
            "year",
            "locality_name",
            "industry_title",
            "annual_avg_emplvl",
            "total_annual_wages",
            "avg_annual_pay",
            "implied_avg_annual_pay",
            "avg_pay_pct_difference"
        ]
    ]
    .head(20)
)

largest_pay_discrepancies

,year,locality_name,industry_title,annual_avg_emplvl,total_annual_wages,avg_annual_pay,implied_avg_annual_pay,avg_pay_pct_difference
11771,2020,Surry County,Unclassified,1.0,26463.0,21170.0,26463.000000,-0.250024
1221,2020,Bath County,Unclassified,2.0,113368.0,48586.0,56684.000000,-0.166674
11772,2021,Surry County,Unclassified,2.0,138902.0,83341.0,69451.000000,0.166665
129,2020,Accomack County,Unclassified,1.0,22771.0,27325.0,22771.000000,0.166661
7273,2020,Louisa County,Educational Services,3.0,71751.0,28700.0,23917.000000,0.166655
8630,2019,Northumberland County,Unclassified,3.0,58521.0,23408.0,19507.000000,0.166652
15113,2021,Lexington city,Administrative and Support and Waste Management,3.0,74785.0,21888.0,24928.333333,-0.138904
13252,2022,Buena Vista city,Unclassified,2.0,19465.0,11123.0,9732.500000,0.125011
15111,2019,Lexington city,Administrative and Support and Waste Management,3.0,61813.0,23180.0,20604.333333,0.111116
14644,2019,Hampton city,"Agriculture, Forestry, Fishing and Hunting",3.0,99600.0,37350.0,33200.000000,0.111111


### Explanation of Large Pay Discrepancies

The largest differences between reported and recomputed average annual pay occur almost exclusively in observations with very small employment counts.

BLS calculates average annual pay using underlying unrounded monthly employment values, while the published annual average employment measure is rounded. For small employment cells, this rounding can produce relatively large percentage differences when annual pay is recomputed from the published values.

These discrepancies are therefore treated as expected rounding effects rather than data-quality failures.

In [34]:
# Summarize missingness across the main analytical variables.

analysis_columns = [
    "annual_avg_estabs",
    "annual_avg_emplvl",
    "total_annual_wages",
    "annual_avg_wkly_wage",
    "avg_annual_pay"
]

missingness_summary = pd.DataFrame({
    "missing_count": panel[analysis_columns].isna().sum(),
    "missing_rate": panel[analysis_columns].isna().mean()
})

missingness_summary

,missing_count,missing_rate
annual_avg_estabs,0,0.000000
annual_avg_emplvl,5289,0.297068
total_annual_wages,5289,0.297068
annual_avg_wkly_wage,5289,0.297068
avg_annual_pay,5289,0.297068


### Missingness Pattern

Missing values in the core employment and wage measures align exactly with the 5,289 disclosure-suppressed QCEW observations.

Establishment counts are complete across the panel, while employment, total wages, average weekly wages, and average annual pay each have a missing rate of approximately **29.7%**.

This indicates that analytical missingness is primarily a known feature of the QCEW confidentiality process rather than evidence of incomplete data ingestion or processing.

In [35]:
# Verify that missing employment values correspond exactly
# to disclosure-suppressed observations.

suppression_missingness_check = pd.crosstab(
    panel["is_suppressed"],
    panel["annual_avg_emplvl"].isna(),
    rownames=["is_suppressed"],
    colnames=["employment_missing"]
)

suppression_missingness_check

employment_missing,False,True
is_suppressed,,
False,12515,0
True,0,5289


### Suppression-Missingness Validation

The cross-tabulation confirms a one-to-one relationship between disclosure suppression and missing employment values.

All 12,515 non-suppressed observations contain published employment data, while all 5,289 suppressed observations have missing employment values. No unexpected missingness appears outside the documented confidentiality process.

This confirms that employment missingness in the analytical panel is systematic and source-driven rather than the result of data-processing errors.

In [36]:
# Inspect records with zero reported establishments.
# -------------------------------------------------
# Establishment counts are complete across the panel, so zero values
# should be examined to determine whether they represent legitimate
# structural records or unexpected observations.

zero_establishment_records = panel[
    panel["annual_avg_estabs"] == 0
].copy()

print(
    f"Records with zero establishments: "
    f"{len(zero_establishment_records):,}"
)

zero_establishment_records[
    [
        "year",
        "locality_name",
        "industry_title",
        "annual_avg_estabs",
        "annual_avg_emplvl",
        "disclosure_code"
    ]
].head(20)

Records with zero establishments: 56


,year,locality_name,industry_title,annual_avg_estabs,annual_avg_emplvl,disclosure_code
8,2023,Accomack County,"Mining, Quarrying, and Oil and Gas Extraction",0,NaN,N
824,2020,Arlington County,"Mining, Quarrying, and Oil and Gas Extraction",0,NaN,N
1104,2023,Bath County,"Mining, Quarrying, and Oil and Gas Extraction",0,NaN,N
1139,2021,Bath County,Transportation and Warehousing,0,NaN,N
1428,2023,Bland County,Real Estate and Rental and Leasing,0,NaN,N
1439,2023,Bland County,Management of Companies and Enterprises,0,NaN,N
2272,2025,Caroline County,Management of Companies and Enterprises,0,NaN,N
2609,2023,Charlotte County,"Mining, Quarrying, and Oil and Gas Extraction",0,NaN,N
2882,2023,Clarke County,Utilities,0,NaN,N
3079,2023,Craig County,Management of Companies and Enterprises,0,NaN,N


### Zero-Establishment Records

A small number of records report zero establishments.

Most of these observations are also disclosure-suppressed, meaning the zero should not be interpreted as evidence of no economic activity. A smaller number of non-suppressed records contain both zero establishments and zero employment, which represent genuine structural zeros in the published data.

Zero-establishment records are therefore retained and interpreted according to disclosure status rather than removed automatically.

In [37]:
# Distinguish suppressed and published zero-establishment records.

zero_establishment_summary = (
    zero_establishment_records
    .assign(
        suppression_status=np.where(
            zero_establishment_records["is_suppressed"],
            "Suppressed",
            "Published"
        )
    )
    ["suppression_status"]
    .value_counts()
    .rename_axis("suppression_status")
    .reset_index(name="record_count")
)

zero_establishment_summary

,suppression_status,record_count
0,Suppressed,54
1,Published,2


### Zero-Establishment Validation

Of the 56 zero-establishment observations, 54 are disclosure-suppressed and 2 are published structural zeros.

This confirms that zero establishment counts do not have a single interpretation across the panel. Suppressed zeros are treated as unavailable information, while published zeros are retained as valid zero-activity observations.

In [38]:
# Inspect the small number of published records with zero establishments.
# ----------------------------------------------------------------------
# These observations are not disclosure-suppressed, so a reported zero
# establishment count can be interpreted as a genuine published zero
# rather than an unavailable confidential value.

published_zero_establishments = (
    zero_establishment_records[
        ~zero_establishment_records["is_suppressed"]
    ]
    [
        [
            "year",
            "locality_name",
            "industry_title",
            "annual_avg_estabs",
            "annual_avg_emplvl",
            "total_annual_wages",
            "annual_avg_wkly_wage",
            "avg_annual_pay",
            "disclosure_code"
        ]
    ]
)

published_zero_establishments

,year,locality_name,industry_title,annual_avg_estabs,annual_avg_emplvl,total_annual_wages,annual_avg_wkly_wage,avg_annual_pay,disclosure_code
6085,2019,Highland County,Unclassified,0,0.0,0.0,0.0,0.0,NaN
6086,2020,Highland County,Unclassified,0,0.0,3432.0,396.0,20592.0,NaN


### Published Zero-Activity Edge Cases

The two published zero-establishment observations occur in Highland County's Unclassified sector.

One record reports zeros across all employment and wage measures. The other reports zero average establishments and employment but positive annual wages, indicating limited activity during the year that is not visible in the rounded annual-average counts.

These records are retained because they are valid published observations. However, zero-employment baselines are excluded from percentage-growth calculations where division by zero would make the result undefined.

In [39]:
# Check for negative values in the main economic measures.
# --------------------------------------------------------
# These variables should not be negative in the QCEW annual panel.
# Any negative observations would require investigation before the
# dataset is used for economic analysis.

economic_measure_columns = [
    "annual_avg_estabs",
    "annual_avg_emplvl",
    "total_annual_wages",
    "annual_avg_wkly_wage",
    "avg_annual_pay"
]

negative_value_summary = pd.DataFrame({
    "negative_count": [
        (panel[column] < 0).sum()
        for column in economic_measure_columns
    ]
}, index=economic_measure_columns)

negative_value_summary

,negative_count
annual_avg_estabs,0
annual_avg_emplvl,0
total_annual_wages,0
annual_avg_wkly_wage,0
avg_annual_pay,0


### Negative-Value Validation

No negative values are present in establishments, employment, total wages, average weekly wages, or average annual pay.

Because these measures should be non-negative by definition, this result provides an additional structural check that the processed panel does not contain obvious invalid numeric values.

In [40]:
# Inspect the largest reported average annual pay values.
# ------------------------------------------------------
# Extremely high wages are not automatically data errors. However,
# unusually large values may reflect small employment cells, highly
# specialized industries, or observations that warrant closer review.

highest_annual_pay = (
    panel[
        panel["avg_annual_pay"].notna()
    ]
    .sort_values(
        "avg_annual_pay",
        ascending=False
    )
    [
        [
            "year",
            "locality_name",
            "industry_title",
            "annual_avg_emplvl",
            "annual_avg_estabs",
            "avg_annual_pay",
            "annual_avg_wkly_wage"
        ]
    ]
    .head(20)
)

highest_annual_pay

,year,locality_name,industry_title,annual_avg_emplvl,annual_avg_estabs,avg_annual_pay,annual_avg_wkly_wage
16412,2019,Portsmouth city,Management of Companies and Enterprises,37.0,6,1103841.0,21228.0
833,2022,Arlington County,Utilities,360.0,12,487911.0,9383.0
834,2023,Arlington County,Utilities,325.0,14,451820.0,8689.0
830,2019,Arlington County,Utilities,355.0,5,429382.0,8257.0
7337,2021,Lunenburg County,Wholesale Trade,96.0,9,426832.0,8208.0
835,2024,Arlington County,Utilities,386.0,16,402836.0,7747.0
832,2021,Arlington County,Utilities,402.0,10,397734.0,7649.0
831,2020,Arlington County,Utilities,387.0,7,396691.0,7629.0
836,2025,Arlington County,Utilities,412.0,18,385773.0,7419.0
6181,2025,Isle of Wight County,Management of Companies and Enterprises,375.0,16,339909.0,6537.0


### Extreme Wage Observations

The highest average annual pay values are concentrated in a small number of industries and localities, particularly Management of Companies and Utilities.

Some observations are supported by relatively large employment bases and persist across multiple years, suggesting that they may reflect genuine high-wage local industry structures. Others occur in much smaller employment cells and warrant closer review.

Extreme wage observations are therefore retained and flagged rather than removed automatically.

In [41]:
# Flag very high average annual pay observations for manual review.
# ---------------------------------------------------------------
# The threshold is used as a diagnostic flag only. Observations above
# the threshold are retained because high wages may be economically valid.

HIGH_PAY_FLAG = 300_000

panel["high_pay_flag"] = (
    panel["avg_annual_pay"].notna()
    & (panel["avg_annual_pay"] >= HIGH_PAY_FLAG)
)

high_pay_flagged = panel[
    panel["high_pay_flag"]
].copy()

print(
    f"Observations with average annual pay >= "
    f"${HIGH_PAY_FLAG:,.0f}: {len(high_pay_flagged):,}"
)

Observations with average annual pay >= $300,000: 13


### High-Pay Diagnostic Flag

Thirteen published observations have average annual pay of at least **$300,000**.

These observations are retained in the analytical dataset. The threshold is used only as a diagnostic flag because unusually high pay may reflect genuine high-wage industries, headquarters activity, specialized employers, or small-cell effects rather than data error.

In [42]:
# Quantify the prevalence of extremely high-pay observations.
# -----------------------------------------------------------
# This provides context for the diagnostic flag by comparing the
# number of flagged records with all observations that have
# published average annual pay.

published_pay_count = (
    panel["avg_annual_pay"]
    .notna()
    .sum()
)

high_pay_flag_rate = (
    len(high_pay_flagged)
    / published_pay_count
)

print(f"Published pay observations: {published_pay_count:,}")
print(f"High-pay observations flagged: {len(high_pay_flagged):,}")
print(f"High-pay flag rate: {high_pay_flag_rate:.2%}")

Published pay observations: 12,515
High-pay observations flagged: 13
High-pay flag rate: 0.10%


### Prevalence of High-Pay Observations

Only **0.10%** of observations with published average annual pay exceed the $300,000 diagnostic threshold.

The very low prevalence indicates that extreme wage values are isolated rather than a broad data-quality concern. These observations are retained and flagged for contextual review rather than excluded from the analytical dataset.

In [43]:
# Inspect the lowest positive average annual pay values.
# -----------------------------------------------------
# Very low reported pay is not automatically invalid, but extreme values
# may reflect very small employment cells, part-year activity, or unusual
# industry-locality combinations that deserve contextual review.

lowest_positive_pay = (
    panel[
        panel["avg_annual_pay"].notna()
        & (panel["avg_annual_pay"] > 0)
    ]
    .sort_values(
        "avg_annual_pay",
        ascending=True
    )
    [
        [
            "year",
            "locality_name",
            "industry_title",
            "annual_avg_emplvl",
            "annual_avg_estabs",
            "avg_annual_pay",
            "annual_avg_wkly_wage"
        ]
    ]
    .head(20)
)

lowest_positive_pay

,year,locality_name,industry_title,annual_avg_emplvl,annual_avg_estabs,avg_annual_pay,annual_avg_wkly_wage
6608,2019,King George County,"Arts, Entertainment, and Recreation",31.0,7,8149.0,157.0
15021,2019,Hopewell city,"Arts, Entertainment, and Recreation",30.0,6,8263.0,159.0
2690,2023,Charlotte County,Educational Services,3.0,15,9571.0,184.0
4704,2019,Giles County,"Arts, Entertainment, and Recreation",20.0,4,9583.0,184.0
16579,2025,Radford city,"Arts, Entertainment, and Recreation",15.0,4,9670.0,186.0
15022,2020,Hopewell city,"Arts, Entertainment, and Recreation",17.0,5,10073.0,194.0
6609,2020,King George County,"Arts, Entertainment, and Recreation",28.0,7,10207.0,196.0
16188,2020,Petersburg city,"Arts, Entertainment, and Recreation",72.0,8,10304.0,198.0
13252,2022,Buena Vista city,Unclassified,2.0,1,11123.0,214.0
16189,2021,Petersburg city,"Arts, Entertainment, and Recreation",71.0,7,11266.0,217.0


### Low-Pay Observation Review

The lowest positive average annual pay values are concentrated primarily in Arts, Entertainment, and Recreation, with additional observations in Accommodation and Food Services and a small number of other sectors.

These values are economically plausible because the affected industries often contain seasonal, part-time, or lower-hour employment. Several observations also occur in relatively small employment cells.

The low-pay observations are therefore retained and interpreted as part of the underlying labor-market structure rather than treated as data-quality errors.

In [44]:
# Calculate establishments per employee for published employment records.
# -----------------------------------------------------------------------
# This diagnostic helps identify locality-industry observations where the
# number of establishments is unusually large relative to employment.
#
# Very high ratios are not automatically errors; they may reflect many
# very small firms, part-year establishments, or unusual reporting patterns.

establishment_ratio_check = panel[
    panel["annual_avg_emplvl"].notna()
    & (panel["annual_avg_emplvl"] > 0)
].copy()

establishment_ratio_check["establishments_per_employee"] = (
    establishment_ratio_check["annual_avg_estabs"]
    / establishment_ratio_check["annual_avg_emplvl"]
)

highest_establishment_ratios = (
    establishment_ratio_check
    .sort_values(
        "establishments_per_employee",
        ascending=False
    )
    [
        [
            "year",
            "locality_name",
            "industry_title",
            "annual_avg_estabs",
            "annual_avg_emplvl",
            "establishments_per_employee"
        ]
    ]
    .head(20)
)

highest_establishment_ratios

,year,locality_name,industry_title,annual_avg_estabs,annual_avg_emplvl,establishments_per_employee
2690,2023,Charlotte County,Educational Services,15,3.0,5.000000
2651,2023,Charlotte County,Information,38,8.0,4.750000
2242,2023,Caroline County,Information,15,4.0,3.750000
97,2023,Accomack County,Educational Services,18,5.0,3.600000
129,2020,Accomack County,Unclassified,3,1.0,3.000000
1689,2023,Brunswick County,Information,12,4.0,3.000000
132,2023,Accomack County,Unclassified,30,12.0,2.500000
96,2022,Accomack County,Educational Services,14,6.0,2.333333
6045,2024,Highland County,"Professional, Scientific, and Technical Services",23,10.0,2.300000
2241,2022,Caroline County,Information,18,8.0,2.250000


### Establishment-to-Employment Ratio Review

The largest establishment-to-employment ratios occur almost entirely in very small employment cells.

Because QCEW establishment counts and employment measures are annual averages, a locality-industry may contain several establishments that operate during only part of the year while reporting a very small annual average employment level.

These observations are therefore retained as plausible edge cases rather than treated as invalid records.

In [45]:
# Flag unusually high establishment-to-employment ratios.
# --------------------------------------------------------
# A ratio above 1 means the reported annual average number of
# establishments exceeds average employment. These cases are retained
# but flagged so their prevalence can be assessed.

HIGH_ESTAB_EMP_RATIO = 1.0

establishment_ratio_check["high_estab_emp_ratio_flag"] = (
    establishment_ratio_check["establishments_per_employee"]
    > HIGH_ESTAB_EMP_RATIO
)

high_ratio_count = (
    establishment_ratio_check["high_estab_emp_ratio_flag"]
    .sum()
)

high_ratio_rate = (
    establishment_ratio_check["high_estab_emp_ratio_flag"]
    .mean()
)

print(f"Records with establishments per employee > 1: {high_ratio_count:,}")
print(f"Share of published positive-employment records: {high_ratio_rate:.2%}")

Records with establishments per employee > 1: 79
Share of published positive-employment records: 0.63%


### Prevalence of High Establishment-to-Employment Ratios

Only **0.63%** of published positive-employment observations have more than one annual-average establishment per employee.

Because these cases are rare and concentrated in very small employment cells, they are treated as unusual but plausible reporting patterns rather than evidence of widespread data-quality issues.

## 6. Consolidated Data-Quality Summary

The preceding checks evaluated structural integrity, disclosure-driven missingness, temporal completeness, extreme year-over-year changes, wage consistency, and unusual establishment-employment relationships.

The following summary consolidates the key diagnostics into a compact quality-control table that can be referenced in later analytical notebooks and project documentation.

In [46]:
# Consolidate the main data-quality diagnostics into one summary table.
# ---------------------------------------------------------------------
# This provides a compact audit of the panel's overall analytical
# readiness without removing valid but unusual observations.

quality_summary = pd.DataFrame({
    "metric": [
        "Total records",
        "Published employment records",
        "Suppressed records",
        "Overall suppression rate",
        "Duplicate locality-industry-year keys",
        "Negative economic values",
        "Series with complete 7-year employment coverage",
        "Series with both 2019 and 2025 employment",
        "Eligible 2019-2025 growth series with >=100 baseline employment",
        "Extreme YoY changes flagged",
        "High-pay observations flagged",
        "High establishment-to-employment ratio records"
    ],
    "value": [
        len(panel),
        panel["annual_avg_emplvl"].notna().sum(),
        panel["is_suppressed"].sum(),
        panel["is_suppressed"].mean(),
        panel.duplicated(
            subset=["area_fips", "industry_code", "year"]
        ).sum(),
        sum(
            (panel[column] < 0).sum()
            for column in economic_measure_columns
        ),
        (series_quality["published_employment_years"] == 7).sum(),
        endpoint_availability["has_2019_and_2025"].sum(),
        len(growth_rank_eligible),
        len(flagged_yoy_changes),
        len(high_pay_flagged),
        high_ratio_count
    ]
})

quality_summary

,metric,value
0,Total records,17804.000000
1,Published employment records,12515.000000
2,Suppressed records,5289.000000
3,Overall suppression rate,0.297068
4,Duplicate locality-industry-year keys,0.000000
5,Negative economic values,0.000000
6,Series with complete 7-year employment coverage,1375.000000
7,Series with both 2019 and 2025 employment,1459.000000
8,Eligible 2019-2025 growth series with >=100 ba...,1145.000000
9,Extreme YoY changes flagged,13.000000


## 7. Save Data-Quality Outputs

The primary quality-control outputs are saved separately from the analytical panel.

These files document temporal completeness, endpoint availability, and diagnostic flags so that subsequent economic analysis can apply consistent eligibility rules without repeating the full quality-assessment workflow.

In [47]:
# Save the locality-industry series quality table.
# ------------------------------------------------
# This table records temporal completeness and analytical-readiness
# classifications for each locality × industry combination.

series_quality_path = (
    OUTPUT_TABLES
    / "locality_industry_series_quality.csv"
)

series_quality.to_csv(
    series_quality_path,
    index=False
)

print(f"Saved series quality table to:\n{series_quality_path}")

Saved series quality table to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\outputs\tables\locality_industry_series_quality.csv


In [48]:
# Save endpoint availability for long-term growth analysis.
# --------------------------------------------------------
# This table identifies whether each locality × industry series
# has published employment values at both the 2019 and 2025 endpoints.

endpoint_availability_path = (
    OUTPUT_TABLES
    / "locality_industry_endpoint_availability.csv"
)

endpoint_availability.to_csv(
    endpoint_availability_path,
    index=False
)

print(f"Saved endpoint availability table to:\n{endpoint_availability_path}")

Saved endpoint availability table to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\outputs\tables\locality_industry_endpoint_availability.csv


In [49]:
# Save the consolidated data-quality summary.
# -------------------------------------------
# This table provides a compact audit of the main structural,
# missingness, completeness, and anomaly-detection results from Notebook 02.

quality_summary_path = (
    OUTPUT_TABLES
    / "data_quality_summary.csv"
)

quality_summary.to_csv(
    quality_summary_path,
    index=False
)

print(f"Saved data-quality summary to:\n{quality_summary_path}")

Saved data-quality summary to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\outputs\tables\data_quality_summary.csv


In [50]:
# Save extreme year-over-year employment changes flagged for review.
# ------------------------------------------------------------------
# These observations are retained in the analytical dataset but exported
# separately so they can be reviewed and discussed in later analysis.

flagged_yoy_path = (
    OUTPUT_TABLES
    / "flagged_yoy_employment_changes.csv"
)

flagged_yoy_changes.to_csv(
    flagged_yoy_path,
    index=False
)

print(f"Saved flagged YoY changes to:\n{flagged_yoy_path}")

Saved flagged YoY changes to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\outputs\tables\flagged_yoy_employment_changes.csv


In [51]:
# Save high-pay observations flagged for contextual review.
# ---------------------------------------------------------
# These records are not removed from the analytical panel. The export
# simply creates a transparent review file for unusually high reported pay.

high_pay_flagged_path = (
    OUTPUT_TABLES
    / "flagged_high_pay_observations.csv"
)

high_pay_flagged.to_csv(
    high_pay_flagged_path,
    index=False
)

print(f"Saved high-pay flagged observations to:\n{high_pay_flagged_path}")

Saved high-pay flagged observations to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\outputs\tables\flagged_high_pay_observations.csv


In [52]:
# Save observations with unusually high establishment-to-employment ratios.
# -------------------------------------------------------------------------
# These records are retained in the analytical dataset but exported
# separately for transparency and potential contextual review.

high_estab_ratio_flagged = (
    establishment_ratio_check[
        establishment_ratio_check["high_estab_emp_ratio_flag"]
    ]
    .copy()
)

high_estab_ratio_path = (
    OUTPUT_TABLES
    / "flagged_high_establishment_employee_ratios.csv"
)

high_estab_ratio_flagged.to_csv(
    high_estab_ratio_path,
    index=False
)

print(
    f"Saved high establishment-to-employment ratio records to:\n"
    f"{high_estab_ratio_path}"
)

Saved high establishment-to-employment ratio records to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\outputs\tables\flagged_high_establishment_employee_ratios.csv


## 8. Save the Quality-Enriched Analytical Panel

The validated QCEW panel is now supplemented with diagnostic variables created during the data-quality assessment.

These flags do not remove or alter valid source observations. Instead, they preserve information about suppression, unusual wage values, and extreme year-over-year changes so that subsequent economic analysis can apply transparent filtering and interpretation rules.

In [53]:
# Save the quality-enriched analytical panel.
# -------------------------------------------
# This version preserves the original economic measures while adding
# diagnostic variables generated during Notebook 02.

quality_enriched_panel_path = (
    PROCESSED_DIR
    / "va_qcew_private_sector_panel_2019_2025_quality_enriched.csv"
)

panel.to_csv(
    quality_enriched_panel_path,
    index=False
)

print(
    f"Saved quality-enriched analytical panel to:\n"
    f"{quality_enriched_panel_path}"
)

Saved quality-enriched analytical panel to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\data\processed\va_qcew_private_sector_panel_2019_2025_quality_enriched.csv


## 9. Data Quality Assessment Summary

The Virginia QCEW industry panel passed the core structural and numeric validation checks required for downstream economic analysis.

Key findings include:

- no duplicate locality-industry-year observations,
- no negative values in the primary economic measures,
- missing employment and wage values are fully explained by documented BLS disclosure suppression,
- approximately 29.7% of panel records are disclosure-suppressed,
- 1,375 locality-industry series contain complete seven-year employment histories,
- 1,459 series contain published employment at both the 2019 and 2025 endpoints,
- 1,145 endpoint-complete series meet the minimum 2019 employment threshold used for percentage-growth rankings,
- only 13 qualified year-over-year employment changes exceed ±100%,
- only 0.10% of published pay observations exceed the $300,000 diagnostic threshold,
- and unusual establishment-to-employment ratios are rare and concentrated in small employment cells.

No observations were removed solely because they appeared unusual. Diagnostic flags were retained so that later analysis can distinguish potentially influential observations from confirmed data-quality problems.

The resulting quality-enriched panel is suitable for regional economic analysis, subject to explicit treatment of disclosure suppression and the eligibility rules established in this notebook.